In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from snowflake.snowpark.context import get_active_session
from snowflake.ml.registry import Registry
from snowflake.ml.model import custom_model

# 1. Load spam.csv with human-readable labels ("Not Spam" and "Spam")
df = pd.read_csv("spam.csv", encoding="latin-1")[["v1", "v2"]].dropna()
df.columns = ["label", "text"]
df["label"] = df["label"].map({"ham": "Not Spam", "spam": "Spam"})

X_train, X_val, y_train, y_val = train_test_split(
    df[["text"]], df["label"], test_size=0.2, random_state=42, stratify=df["label"]
)

# 2. Train pipeline on text labels directly
spam_pipeline = Pipeline(steps=[
    ("preprocessor", ColumnTransformer([("tfidf", TfidfVectorizer(stop_words="english"), "text")])),
    ("classifier", LogisticRegression(max_iter=1000, random_state=42))
])
spam_pipeline.fit(X_train, y_train)

# 3. Wrap in a CustomModel so the output column is cleanly named "RESULT"
class HumanReadableSpamClassifier(custom_model.CustomModel):
    def __init__(self, context: custom_model.ModelContext) -> None:
        super().__init__(context)

    @custom_model.inference_api
    def predict(self, input_df: pd.DataFrame) -> pd.DataFrame:
        preds = self.context["model"].predict(input_df[["text"]])
        return pd.DataFrame({"RESULT": preds})

ctx = custom_model.ModelContext(model=spam_pipeline)
wrapped_model = HumanReadableSpamClassifier(ctx)

# 4. Set active database/schema and log as V2 in Snowflake Model Registry
session = get_active_session()
session.sql("USE DATABASE SPAM_ML_DB").collect()
session.sql("USE SCHEMA MODELS").collect()

reg = Registry(session=session, database_name="SPAM_ML_DB", schema_name="MODELS")

v2 = reg.log_model(
    model=wrapped_model,
    model_name="SMS_SPAM_MODEL",
    version_name="V2",
    sample_input_data=X_train.head(5),
    comment="Human-readable SMS Spam Classifier returning 'Spam' or 'Not Spam'"
)
print("Registered V2! Test it in the Playground now.")

In [ ]:
# 1. Print the live public REST API URL
services_df = model_version.list_services()
display(services_df)
print("Live Endpoint:", f"https://{services_df.loc[0, 'inference_endpoint']}/predict")

# 2. Test with your own custom message
my_msg = pd.DataFrame({"text": ["WIN a brand new iPhone! Click here now to claim your prize!"]})
pred = model_version.run(my_msg, function_name="predict", service_name="SMS_SPAM_SERVICE")
prob = model_version.run(my_msg, function_name="predict_proba", service_name="SMS_SPAM_SERVICE")

my_msg["Prediction"] = pred["output_feature_0"].map({0: "Ham (Not Spam)", 1: "Spam"})
my_msg["Spam_Probability"] = prob["output_feature_1"].round(4)
display(my_msg)

In [ ]:
model_version.delete_service("SMS_SPAM_SERVICE")
print("SMS_SPAM_SERVICE deleted! Your trained model remains safely saved in SPAM_ML_DB.MODELS.SMS_SPAM_MODEL (V1).")